# DBSCAN Demystified: Epsilon, Min Samples, Noise Detection and Arbitrary Shapes

---

**Author:** [Your Name]  
**GitHub:** [https://github.com/YOUR_USERNAME/dbscan-tutorial](https://github.com/YOUR_USERNAME/dbscan-tutorial)  

**References:**
- Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *KDD-96*, 226–231.
- Schubert, E. et al. (2017). DBSCAN Revisited, Revisited. *ACM TODS*, 42(3). https://doi.org/10.1145/3068335
- Scikit-learn DBSCAN documentation: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html
- Rahmah, N. & Sitanggang, I.S. (2016). Determination of Optimal Epsilon (Eps) Value on DBSCAN Algorithm. *IOP Conf. Series*. https://doi.org/10.1088/1755-1315/31/1/012012

---

## Overview

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) is a clustering algorithm that groups points based on **local density** rather than distance to centroids. Unlike K-Means, DBSCAN:

- Discovers clusters of **arbitrary shape** (crescents, rings, spirals)
- **Automatically detects noise/outliers** (labelled as -1)
- Does **not require the number of clusters** to be specified in advance

This tutorial experiments with:

| Parameter | What it controls |
|---|---|
| `eps` (ε) | Neighbourhood radius — how close two points must be to be considered neighbours |
| `min_samples` | Minimum neighbours within ε to qualify a point as a core point |

## 1. Setup and Imports

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons, make_blobs, make_circles
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

# Reproducibility
SEED = 42
np.random.seed(SEED)

# Colour-blind friendly palette (Wong 2011)
CB = {
    'blue':   '#0072B2',
    'amber':  '#E69F00',
    'green':  '#009E73',
    'red':    '#D55E00',
    'purple': '#CC79A7',
    'sky':    '#56B4E9',
    'yellow': '#F0E442',
    'noise':  '#BBBBBB',
}
CLUSTER_COLORS = [CB['blue'], CB['amber'], CB['green'], CB['red'],
                  CB['purple'], CB['sky'], CB['yellow']]

plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 11,
    'axes.titlesize': 12,
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('All imports successful.')

## 2. How DBSCAN Works

DBSCAN classifies every point as one of three types:

- **Core point**: has at least `min_samples` neighbours within radius `eps`
- **Border point**: within `eps` of a core point, but fewer than `min_samples` neighbours itself
- **Noise point**: not reachable from any core point — labelled **-1**

Clusters are formed by connecting core points that are within `eps` of each other, then attaching border points to the nearest cluster.

**Algorithm steps:**
1. For each unvisited point p, find all neighbours within ε
2. If |neighbours| ≥ min_samples → p is a core point → start a new cluster
3. Expand cluster: recursively add all density-reachable points
4. If |neighbours| < min_samples → mark as noise (may be re-assigned if reached by another core point)
5. Repeat until all points are visited

In [ ]:
# Visualise core / border / noise concept
fig, ax = plt.subplots(figsize=(8, 6))
np.random.seed(7)

pts = np.array([
    [2, 2],[2.3, 2.1],[1.8, 2.4],[2.2, 1.8],[2.5, 2.3],[1.9, 1.9],  # core region
    [3.2, 2.8],[3.0, 2.6],                                             # border
    [5.5, 4.5],[1.0, 5.0],[4.8, 1.2],                                 # noise
])
pt_labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, -1, -1, -1])
colors = [CLUSTER_COLORS[l] if l >= 0 else CB['noise'] for l in pt_labels]

# epsilon circle around the core point
circle = plt.Circle(pts[0], 0.9, fill=False, color=CB['blue'],
                     linestyle='--', linewidth=1.8, alpha=0.7)
ax.add_patch(circle)
ax.text(pts[0][0] + 0.92, pts[0][1] + 0.05, r'$\varepsilon$', fontsize=14, color=CB['blue'])

ax.scatter(pts[:, 0], pts[:, 1], c=colors, s=120, edgecolors='white', linewidth=1.5, zorder=5)

ax.annotate('Core point\n(5 neighbours in ε)', xy=pts[0], xytext=(0.5, 3.4),
            arrowprops=dict(arrowstyle='->', color=CB['blue'], lw=1.5), fontsize=9, color=CB['blue'])
ax.annotate('Border point', xy=pts[6], xytext=(3.8, 3.6),
            arrowprops=dict(arrowstyle='->', color=CB['amber'], lw=1.5), fontsize=9, color=CB['amber'])
ax.annotate('Noise point', xy=pts[8], xytext=(4.5, 5.2),
            arrowprops=dict(arrowstyle='->', color='#888888', lw=1.5), fontsize=9, color='#888888')

legend_els = [
    mpatches.Patch(color=CB['blue'],  label='Core point (cluster member)'),
    mpatches.Patch(color=CB['amber'], label='Border point (reachable)'),
    mpatches.Patch(color=CB['noise'], label='Noise point (outlier, label=-1)'),
]
ax.legend(handles=legend_els, loc='lower right', fontsize=9)
ax.set_xlim(0, 7); ax.set_ylim(0.5, 6.5)
ax.set_title('DBSCAN Point Classification: Core, Border, and Noise')
ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')
plt.tight_layout()
plt.savefig('dbscan_fig1_concept.png', bbox_inches='tight')
plt.show()

## 3. Dataset Generation

We use three synthetic datasets that showcase DBSCAN's strengths:

| Dataset | Shape | Why useful |
|---|---|---|
| Two Moons | Crescent shapes | Non-convex — K-Means fails |
| Concentric Rings | Nested circles | Non-convex, uniform density |
| Blobs + Noise | Compact clusters + outliers | Noise detection showcase |

In [ ]:
# Generate and standardise datasets
X_moons, y_moons = make_moons(n_samples=300, noise=0.07, random_state=SEED)
X_moons = StandardScaler().fit_transform(X_moons)

X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.4, random_state=SEED)
X_circles = StandardScaler().fit_transform(X_circles)

X_blobs_clean, _ = make_blobs(n_samples=250, centers=3, cluster_std=0.5, random_state=SEED)
X_noise_pts = np.random.uniform(-7, 7, size=(50, 2))
X_noisy = StandardScaler().fit_transform(np.vstack([X_blobs_clean, X_noise_pts]))

print(f'Two Moons   : {X_moons.shape}')
print(f'Rings       : {X_circles.shape}')
print(f'Blobs+Noise : {X_noisy.shape} (50 injected outliers)')

## 4. Selecting ε: The k-Distance Plot

Choosing a good `eps` is the most important step. The **k-distance plot** provides a principled method:

1. For each point, compute its distance to its k-th nearest neighbour (k = min_samples)
2. Sort these distances in descending order and plot them
3. The **elbow** (sharp change in gradient) indicates a good ε — beyond this, distances grow rapidly, meaning points are likely noise

In [ ]:
k = 5
nbrs = NearestNeighbors(n_neighbors=k).fit(X_moons)
distances, _ = nbrs.kneighbors(X_moons)
k_distances = np.sort(distances[:, k - 1])[::-1]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(k_distances, color=CB['blue'], linewidth=2)
ax.axhline(y=0.35, color=CB['red'], linestyle='--', linewidth=1.5, label='Suggested ε ≈ 0.35 (elbow)')
ax.fill_between(range(len(k_distances)), k_distances, 0, alpha=0.08, color=CB['blue'])
ax.set_xlabel('Points sorted by k-distance (descending)')
ax.set_ylabel(f'{k}-NN distance')
ax.set_title(f'k-Distance Plot (k={k}) — Elbow Indicates Optimal ε')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('dbscan_fig7_kdistance.png', bbox_inches='tight')
plt.show()
print(f'Suggested eps from elbow: ~0.35')

## 5. Experiment 1 — Effect of ε (epsilon)

We sweep ε from 0.1 to 1.0 with `min_samples=5` fixed, observing:
- **Too small ε**: almost every point becomes noise
- **Optimal ε (~0.35)**: both moons correctly identified
- **Too large ε**: clusters merge into one

In [ ]:
eps_vals = [0.1, 0.2, 0.35, 0.6, 1.0]

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, eps in zip(axes, eps_vals):
    db = DBSCAN(eps=eps, min_samples=5).fit(X_moons)
    labels = db.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    colors     = [CLUSTER_COLORS[l % len(CLUSTER_COLORS)] if l >= 0 else CB['noise'] for l in labels]
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=colors, s=18, alpha=0.8)
    ax.set_title(f'ε={eps}\n{n_clusters} clusters, {n_noise} noise', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Effect of ε — Two Moons Dataset (min_samples=5)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('dbscan_fig2_epsilon_sweep.png', bbox_inches='tight')
plt.show()

## 6. Experiment 2 — Effect of min_samples

We fix `eps=0.35` and vary `min_samples` from 2 to 40:
- **Small min_samples (2–5)**: very few noise points — even isolated points form clusters
- **Moderate min_samples (5–10)**: balanced — outliers correctly labelled as noise
- **Large min_samples (20+)**: over-restrictive — many genuine cluster points become noise

In [ ]:
minpts_vals = [2, 5, 10, 20, 40]

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
for ax, mpts in zip(axes, minpts_vals):
    db = DBSCAN(eps=0.35, min_samples=mpts).fit(X_moons)
    labels = db.labels_
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    colors     = [CLUSTER_COLORS[l % len(CLUSTER_COLORS)] if l >= 0 else CB['noise'] for l in labels]
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=colors, s=18, alpha=0.8)
    ax.set_title(f'min_samples={mpts}\n{n_clusters} clusters, {n_noise} noise', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Effect of min_samples — Two Moons Dataset (ε=0.35)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('dbscan_fig3_minsamples_sweep.png', bbox_inches='tight')
plt.show()

## 7. Experiment 3 — Parameter Sensitivity Heatmap

A systematic grid search over ε × min_samples shows how both parameters jointly control the number of clusters found and the number of points labelled as noise.

In [ ]:
eps_range  = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.8]
mpts_range = [2, 3, 5, 8, 10, 15, 20]

nc_grid = np.zeros((len(mpts_range), len(eps_range)))
nn_grid = np.zeros((len(mpts_range), len(eps_range)))

for i, mpts in enumerate(mpts_range):
    for j, eps in enumerate(eps_range):
        db = DBSCAN(eps=eps, min_samples=mpts).fit(X_moons)
        lbl = db.labels_
        nc_grid[i, j] = len(set(lbl)) - (1 if -1 in lbl else 0)
        nn_grid[i, j] = (lbl == -1).sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im1 = axes[0].imshow(nc_grid, aspect='auto', cmap='YlOrRd', origin='lower')
axes[0].set_xticks(range(len(eps_range)))
axes[0].set_xticklabels(eps_range, rotation=45, fontsize=8)
axes[0].set_yticks(range(len(mpts_range)))
axes[0].set_yticklabels(mpts_range, fontsize=8)
axes[0].set_xlabel('ε (epsilon)'); axes[0].set_ylabel('min_samples')
axes[0].set_title('Number of Clusters Found')
for i in range(len(mpts_range)):
    for j in range(len(eps_range)):
        axes[0].text(j, i, int(nc_grid[i, j]), ha='center', va='center',
                     fontsize=7, color='white' if nc_grid[i, j] > 3 else 'black')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

im2 = axes[1].imshow(nn_grid, aspect='auto', cmap='Blues', origin='lower')
axes[1].set_xticks(range(len(eps_range)))
axes[1].set_xticklabels(eps_range, rotation=45, fontsize=8)
axes[1].set_yticks(range(len(mpts_range)))
axes[1].set_yticklabels(mpts_range, fontsize=8)
axes[1].set_xlabel('ε (epsilon)'); axes[1].set_ylabel('min_samples')
axes[1].set_title('Number of Noise Points')
for i in range(len(mpts_range)):
    for j in range(len(eps_range)):
        axes[1].text(j, i, int(nn_grid[i, j]), ha='center', va='center',
                     fontsize=7, color='white' if nn_grid[i, j] > 150 else 'black')
plt.colorbar(im2, ax=axes[1], shrink=0.8)

plt.suptitle('Parameter Sensitivity Heatmap: ε vs min_samples (Two Moons)', fontsize=12)
plt.tight_layout()
plt.savefig('dbscan_fig6_heatmap.png', bbox_inches='tight')
plt.show()
print('Optimal region (2 clusters, low noise): eps~0.3-0.4, min_samples~5-8')

## 8. Experiment 4 — Arbitrary Shape Detection: DBSCAN vs K-Means

K-Means assumes spherical, similarly-sized clusters and uses Euclidean distance to centroids. This causes it to fail on any non-convex geometry. DBSCAN uses **local density** and requires no shape assumptions.

In [ ]:
shapes = {
    'Two Moons': (X_moons, y_moons),
    'Concentric Rings': (X_circles, y_circles),
    'Anisotropic Blobs': (
        lambda: (
            StandardScaler().fit_transform(
                np.dot(make_blobs(n_samples=300, centers=3, random_state=SEED)[0],
                       [[0.6, -0.6], [0.4, 0.8]])
            ),
            make_blobs(n_samples=300, centers=3, random_state=SEED)[1]
        )
    )(),
}

dbscan_params = {
    'Two Moons':         dict(eps=0.35, min_samples=5),
    'Concentric Rings':  dict(eps=0.25, min_samples=5),
    'Anisotropic Blobs': dict(eps=0.6,  min_samples=5),
}
k_params = {'Two Moons': 2, 'Concentric Rings': 2, 'Anisotropic Blobs': 3}

fig, axes = plt.subplots(3, 3, figsize=(13, 11))

for row, (name, (X, y_true)) in enumerate(shapes.items()):
    n_k = k_params[name]

    # Ground truth
    true_colors = [CLUSTER_COLORS[c % len(CLUSTER_COLORS)] for c in y_true]
    axes[row, 0].scatter(X[:, 0], X[:, 1], c=true_colors, s=18, alpha=0.8)
    axes[row, 0].set_title(f'{name}\nGround Truth', fontsize=9)
    axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])

    # K-Means
    km = KMeans(n_clusters=n_k, random_state=SEED, n_init=10).fit(X)
    km_colors = [CLUSTER_COLORS[c % len(CLUSTER_COLORS)] for c in km.labels_]
    axes[row, 1].scatter(X[:, 0], X[:, 1], c=km_colors, s=18, alpha=0.8)
    axes[row, 1].set_title(f'K-Means (k={n_k})\nFails on non-convex shapes', fontsize=9)
    axes[row, 1].set_xticks([]); axes[row, 1].set_yticks([])

    # DBSCAN
    db = DBSCAN(**dbscan_params[name]).fit(X)
    lbl = db.labels_
    n_clusters = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_noise    = (lbl == -1).sum()
    db_colors  = [CLUSTER_COLORS[l % len(CLUSTER_COLORS)] if l >= 0 else CB['noise'] for l in lbl]
    axes[row, 2].scatter(X[:, 0], X[:, 1], c=db_colors, s=18, alpha=0.8)
    axes[row, 2].set_title(f'DBSCAN: {n_clusters} clusters, {n_noise} noise\nCaptures arbitrary shapes', fontsize=9)
    axes[row, 2].set_xticks([]); axes[row, 2].set_yticks([])

plt.suptitle('DBSCAN vs K-Means: Arbitrary Shape Detection', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('dbscan_fig4_shapes.png', bbox_inches='tight')
plt.show()

## 9. Experiment 5 — Noise Detection

We inject 50 uniformly random outlier points into a compact 3-cluster dataset to demonstrate how epsilon and min_samples jointly control the noise detection sensitivity.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Raw data
axes[0].scatter(X_noisy[:, 0], X_noisy[:, 1], c=CB['blue'], s=22, alpha=0.6)
axes[0].set_title('Raw Data\n(250 cluster pts + 50 random outliers)', fontsize=9)
axes[0].set_xticks([]); axes[0].set_yticks([])

# DBSCAN – well-tuned
db_good = DBSCAN(eps=0.5, min_samples=8).fit(X_noisy)
lbl = db_good.labels_
n_cl = len(set(lbl)) - (1 if -1 in lbl else 0)
n_ns = (lbl == -1).sum()
colors = [CLUSTER_COLORS[l % len(CLUSTER_COLORS)] if l >= 0 else CB['noise'] for l in lbl]
axes[1].scatter(X_noisy[:, 0], X_noisy[:, 1], c=colors, s=22, alpha=0.8)
noise_mask = lbl == -1
axes[1].scatter(X_noisy[noise_mask, 0], X_noisy[noise_mask, 1],
                c=CB['noise'], s=60, marker='x', linewidths=1.8,
                label=f'Detected noise ({n_ns} pts)', zorder=6)
axes[1].set_title(f'DBSCAN ε=0.5, min_samples=8\n{n_cl} clusters, {n_ns} noise detected', fontsize=9)
axes[1].legend(fontsize=8); axes[1].set_xticks([]); axes[1].set_yticks([])

# DBSCAN – too loose
db_loose = DBSCAN(eps=1.5, min_samples=3).fit(X_noisy)
lbl2 = db_loose.labels_
n_cl2 = len(set(lbl2)) - (1 if -1 in lbl2 else 0)
n_ns2 = (lbl2 == -1).sum()
colors2 = [CLUSTER_COLORS[l % len(CLUSTER_COLORS)] if l >= 0 else CB['noise'] for l in lbl2]
axes[2].scatter(X_noisy[:, 0], X_noisy[:, 1], c=colors2, s=22, alpha=0.8)
axes[2].set_title(f'DBSCAN ε=1.5, min_samples=3\n{n_cl2} clusters, {n_ns2} noise — over-merged', fontsize=9)
axes[2].set_xticks([]); axes[2].set_yticks([])

plt.suptitle('Noise Detection: ε and min_samples Control Outlier Sensitivity', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('dbscan_fig5_noise.png', bbox_inches='tight')
plt.show()

print(f'Well-tuned: {n_cl} clusters, {n_ns} noise points detected (out of 50 injected)')
print(f'Over-loose: {n_cl2} clusters, {n_ns2} noise points (outliers absorbed into clusters)')

## 10. Summary Table

In [ ]:
summary = [
    {'Experiment': 'ε sweep (Fig 2)',       'Dataset': 'Two Moons', 'Finding': 'ε≈0.35 optimal; too small→all noise, too large→merged clusters'},
    {'Experiment': 'min_samples sweep (Fig 3)', 'Dataset': 'Two Moons', 'Finding': 'min_samples=5 balanced; >20 turns cluster points into noise'},
    {'Experiment': 'Heatmap (Fig 6)',        'Dataset': 'Two Moons', 'Finding': 'Optimal region: ε=0.3–0.4, min_samples=5–8'},
    {'Experiment': 'Shape detection (Fig 4)', 'Dataset': 'Moons/Rings/Aniso', 'Finding': 'DBSCAN succeeds where K-Means fails on non-convex shapes'},
    {'Experiment': 'Noise detection (Fig 5)', 'Dataset': 'Blobs + outliers', 'Finding': 'ε=0.5, min=8 detects ~50 noise pts; ε=1.5, min=3 absorbs them'},
]
pd.DataFrame(summary)

## 11. Key Takeaways

| Finding | Practical advice |
|---|---|
| ε controls cluster granularity | Use the k-distance elbow plot to select ε objectively |
| min_samples controls noise sensitivity | Rule of thumb: min_samples ≥ dimensionality + 1 (often 5 works well) |
| Both parameters interact | Always inspect the heatmap; they are not independent |
| DBSCAN handles arbitrary shapes | Unlike K-Means, no assumptions about cluster geometry |
| Always standardise features | DBSCAN is distance-based — unscaled features distort ε equally |

---

## References

1. Ester, M., Kriegel, H. P., Sander, J., & Xu, X. (1996). A density-based algorithm for discovering clusters in large spatial databases with noise. *KDD-96*, 226–231.
2. Schubert, E. et al. (2017). DBSCAN Revisited, Revisited. *ACM TODS*, 42(3). https://doi.org/10.1145/3068335
3. Rahmah, N. & Sitanggang, I.S. (2016). Determination of Optimal Epsilon (Eps) Value on DBSCAN Algorithm. *IOP Conf. Series*. https://doi.org/10.1088/1755-1315/31/1/012012
4. Scikit-learn DBSCAN documentation: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html